In [1]:
import requests
import csv
import json

In [2]:
baseURL = "https://www.loc.gov/"
endpoint = "free-to-use/"
collection = "libraries/"
parameters = {"fo":"json"}

In [3]:
collection_list = requests.get(baseURL + endpoint + collection, params=parameters)
with open("ftu-libraries-set-info.json", "w", encoding="utf-8") as f:
        f.write(json.dumps(collection_list.json(), indent=2))

In [4]:
headers = ["image", "link", "title"]

with open("ftu-libraries-set-list.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for item in collection_list.json()["content"]["components"][0]["masonry_gallery"][0]["items"]:
        item["title"] = item["title"].rstrip()
        writer.writerow(item)
print(f"Wrote {len(collection_list.json()['content']['components'][0]['masonry_gallery'][0]['items'])} rows to ftu-libraries-set-list.csv")

Wrote 62 rows to ftu-libraries-set-list.csv


In [5]:
from pathlib import Path
import glob
import os

In [6]:
def regenerate_collection_list(collection_csv):
    coll_items = []
    with open(collection_csv, "r", newline="", encoding="utf-8") as f:
        data = csv.DictReader(f)
        for row in data:
            row_dict = dict(row)
            for field in data.fieldnames:
                row_dict[field] = row[field]
            coll_items.append(row_dict)
        return coll_items

In [7]:
collection_csv = Path("ftu-libraries-set-list.csv")
collection_set_list = regenerate_collection_list(collection_csv)

In [8]:
baseURL = "https://www.loc.gov"
parameters = {"fo":"json"}

In [9]:
item_metadata_directory = Path("item-metadata").mkdir(exist_ok=True)

In [10]:
item_count = 0
error_count = 0
file_count = 0

item_metadata_file_prefix = "item-metadata"
json_suffix = ".json"

for item in collection_set_list:
    if item["link"] == "link":
        continue
    if "?" in item["link"]:
        resource_ID = item["link"]
        short_ID = item["link"].split("/")[2]
        item_metadata = requests.get(baseURL + resource_ID, params=parameters)
        print("Requested", item_metadata.url, item_metadata.status_code)
        if item_metadata.status_code != 200:
            print("Requested", item_metadata.url, item_metadata.status_code)
            error_count += 1
            continue
        try:
            item_metadata.json()
        except:
            error_count += 1
            print("No json found")
            continue
        fout = os.path.join(
            "item-metadata", item_metadata_file_prefix + "-" + short_ID + json_suffix
        )
        with open(fout, "w", encoding="utf-8") as f:
            f.write(json.dumps(item_metadata.json()["item"], indent=2))
            file_count += 1
            print("Wrote", fout)
        item_count += 1
    else:
        resource_ID = item["link"]
        short_ID = item["link"].split("/")[2]
        item_metadata = requests.get(baseURL + resource_ID, params=parameters)
        print("Requested", item_metadata.url, item_metadata.status_code)
        if item_metadata.status_code != 200:
            print("Requested", item_metadata.url, item_metadata.status_code)
            error_count += 1
            continue
        try:
            item_metadata.json()
        except:
            error_count += 1
            print("No json found")
            continue
        fout = os.path.join(
            "item-metadata", item_metadata_file_prefix + "-" + short_ID + json_suffix
        )
        with open(fout, "w", encoding="utf-8") as f:
            f.write(json.dumps(item_metadata.json()["item"], indent=2))
            file_count += 1
            print("Wrote", fout)
        item_count += 1

print("Items requested:", item_count)
print("Errors:", error_count)
print("Files written:", file_count)

Requested https://www.loc.gov/resource/cph.3f05183/?fo=json 200
Wrote item-metadata/item-metadata-cph.3f05183.json
Requested https://www.loc.gov/resource/highsm.20336/?fo=json 200
Wrote item-metadata/item-metadata-highsm.20336.json
Requested https://www.loc.gov/resource/fsa.8d24709/?fo=json 200
Wrote item-metadata/item-metadata-fsa.8d24709.json
Requested https://www.loc.gov/resource/highsm.36052/?fo=json 200
Wrote item-metadata/item-metadata-highsm.36052.json
Requested https://www.loc.gov/resource/highsm.51772/?fo=json 200
Wrote item-metadata/item-metadata-highsm.51772.json
Requested https://www.loc.gov/resource/cph.3b43255/?fo=json 200
Wrote item-metadata/item-metadata-cph.3b43255.json
Requested https://www.loc.gov/resource/highsm.20483/?fo=json 200
Wrote item-metadata/item-metadata-highsm.20483.json
Requested https://www.loc.gov/resource/highsm.29207/?fo=json 200
Wrote item-metadata/item-metadata-highsm.29207.json
Requested https://www.loc.gov/resource/fsa.8b32222/?fo=json 200
Wrote 

In [11]:
files_directory = Path("item-files").mkdir(exist_ok=True)

In [12]:
search_for_metadata = Path("item-metadata")
metadata_file_list = list(search_for_metadata.glob("*.json"))

In [13]:
item_image_urls = []
count = 0

for item in metadata_file_list:
    with open(item, "r", encoding="utf-8") as f:
        metadata = json.load(f)
        image_url_no = len(metadata["image_url"])
        image_url = metadata["image_url"][-1]
        item_image_urls.append(image_url)
        count += 1

print(f"Identified {count} image URLS")

Identified 82 image URLS


In [14]:
collection_set_list_with_images = []

for item in metadata_file_list:
    with open(item, "r", encoding="utf-8") as f:
        item_metadata = json.load(f)
        item_metadata_dict = dict(item_metadata)
        item_metadata_dict["item_URI"] = item_metadata["id"]
        try:
            item_metadata_dict["lccn"] = item_metadata["library_of_congress_control_number"]
        except:
            item_metadata_dict["lccn"] = None
        item_metadata_dict["title"] = item_metadata["title"]
        item_metadata_dict["image_URL_large"] = item_metadata["image_url"][-1]
        collection_set_list_with_images.append(item_metadata_dict)


In [16]:
item_count = 0
error_count = 0
file_count = 0

img_file_prefix = "item"
img_file_suffix = "image.jpg"

for item in collection_set_list_with_images:
    image_URL = item["image_URL_large"]
    short_ID = item["item_URI"].split("/")[4]
    print("Requested", image_URL)
    item_count += 1
    r = requests.get(image_URL)
    if r.status_code == 200:
        img_out = os.path.join("item-files", img_file_prefix + "-" + short_ID + "-" + img_file_suffix)
        with open(img_out, "wb") as f:
            f.write(r.content)
            print("Wrote", img_out)
            file_count += 1

print("Files requested:", item_count)
print("Errors:", error_count)
print("Files written:", file_count)

Requested https://tile.loc.gov/storage-services/service/pnp/habshaer/ks/ks0000/ks0072/photos/069905pv.jpg#h=815&w=1024
Wrote item-files/item-ks0072-image.jpg
Requested https://tile.loc.gov/image-services/iiif/service:pnp:highsm:36000:36052/full/pct:25/0/default.jpg#h=1448&w=2172
Wrote item-files/item-2017686535-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/cph/3f00000/3f05000/3f05100/3f05168v.jpg#h=1024&w=708
Wrote item-files/item-98508385-image.jpg
Requested https://tile.loc.gov/image-services/iiif/service:pnp:highsm:36000:36052/full/pct:25/0/default.jpg#h=1448&w=2172
Wrote item-files/item-2017686535-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/highsm/60200/60215v.jpg#h=741&w=1024
Wrote item-files/item-2020723718-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp/habshaer/hi/hi0100/hi0135/photos/058276pv.jpg#h=733&w=1024
Wrote item-files/item-hi0135-image.jpg
Requested https://tile.loc.gov/storage-services/service/pnp